<img src="../../../Individual-Contest/Restroom/figs/IOAI-Logo.png" alt="Logo IOAI" width="200" height="auto">

[IOAI 2025 (Pekin, Chine), concours individuel](https://ioai-official.org/china-2025)

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SKonteye/IOAI-2025/blob/main/fr/Individual-Contest/Restroom/Restroom.ipynb)


# Appariement d'icones de toilettes

## 1. Description du probleme

Cette tache vise a entrainer un modele a apprendre la relation entre les **icones masculine et feminine des toilettes provenant des memes toilettes**. A partir de donnees d'entrainement etiquetees, vous devez entrainer un modele d'appariement qui, etant donne une image requete, retrouve son **homologue de genre oppose** (par exemple, faire correspondre une icone masculine dans son image recadree a son homologue feminin dans son image originale), sous la contrainte que les deux icones proviennent des **memes toilettes**.

Par exemple, pour l'image requete (une icone masculine recadree) presentee ci-dessous dans la Fig.1, son homologue correspondant apparait dans la Fig.2 (une icone feminine dans l'image originale provenant des memes toilettes).

| <img src="../../../Individual-Contest/Restroom/figs/Restroom Fig 1.png" height="200"> | <img src="../../../Individual-Contest/Restroom/figs/Restroom Fig 2.png" height="200"> |
| ------------------------------------------------------------ | ------------------------------------------------------------ |



## 2. Jeu de donnees

**(1) Ensemble d'entrainement :**

Utilise pour l'entrainement du modele. Structure du repertoire :

```bash
train/
├── crop/         # Icones recadrees
│   ├── female/   1.png, 2.png, ...
│   └── male/
└── orig/         # Icones originales
    ├── female/
    └── male/
```

Chaque sous-dossier contient des icones nommees de `1.png` a `82.png`, ou le numero designe l'ID des toilettes. Pour chaque toilette, il y a quatre images (notez que quatre images des memes toilettes partagent un meme ID unique dans les quatre sous-dossiers) :

- `crop/female/i.png` -> icone feminine recadree
- `crop/male/i.png`  -> icone masculine recadree
- `orig/female/i.png` -> icone feminine originale
- `orig/male/i.png`  -> icone masculine originale

**(2) Ensemble de validation et ensemble de test :**

L'ensemble de validation (test_a) et l'ensemble de test (test_b) contiennent chacun deux sous-dossiers :

```bash
test_a/           (ou test_b/)
├── query/        # Icones recadrees a apparier
└── gallery/      # Icones originales candidates
```

- `query/` : icones **recadrees** a apparier.
- `gallery/` : ensemble d'icones **originales** parmi lesquelles trouver la correspondance.
- Remarques :
    - Contrairement au split d'entrainement, les noms de fichiers dans `query/` et `gallery/` sont numerotes independamment et melanges, ce qui signifie que l'appariement ne peut pas reposer sur les IDs.
    - Pour chaque icone recadree de `query/`, il existe exactement deux originales dans `gallery/` (une masculine, une feminine, provenant des memes toilettes), c'est-a-dire $\text{len}(\text{gallery})=2 * \text{len}(\text{query})$.
- Toutes les images sont au format `.png`.
- L'ensemble de validation contient $10$ images dans son dossier `query/`, et l'ensemble de test en contient $30$.

## 3. Tache

Pour chaque image du dossier `query/`, predisez l'image de `gallery/` qui :
- est du **sexe oppose**, et
- provient des **memes toilettes**

Cet appariement doit etre realise **a l'aide de votre modele entraine**.

## 4. Exigences de soumission

Vous devez soumettre un notebook nomme `submission.ipynb`, contenant :

- Le processus d'entrainement du modele (avec les donnees `train/`)
- Le processus d'appariement pour les deux ensembles de test (`test_a/` et `test_b/`)
- Le notebook doit produire deux fichiers `.npy` :
```bash
submission_a.npy     # Resultats d'appariement pour l'ensemble de validation (test_a).
submission_b.npy     # Resultats d'appariement pour l'ensemble de test (test_b).
```

Chaque fichier npy est un tableau unidimensionnel dont la taille est egale au nombre de requetes, et chaque valeur du tableau correspond a l'ID de l'image dans la gallery.

Par exemple, un `submission_a.npy` valide peut ressembler a ceci :
<img src="../../../Individual-Contest/Restroom/figs/Restroom Fig 3.png" width="300">

Vous trouverez egalement un exemple de format de sortie dans [baseline.ipynb](https://ioai.bohrium.com/notebooks/81153159178).

## 5. Score

- Si la soumission se termine dans le temps imparti, le score est calcule comme :
  $$
  \text{Score} = \frac{\text{Nombre d'appariements corrects}}{\text{Nombre total de requetes}}
  $$

  un nombre reel compris entre $0.0$ et $1.0$ (inclus).

- Les soumissions qui depassent la limite de temps recoivent un score de **0**.

## 6. Baseline et ensemble d'entrainement

- Vous trouverez ci-dessous la solution baseline.
- Le jeu de donnees se trouve dans le dossier `training_set`.
- Le meilleur score du comite scientifique pour cette tache est $0.90$ sur le classement B ; ce score est utilise pour l'unification des scores.
- Le score baseline du comite scientifique pour cette tache est $0.77$ sur le classement B ; ce score est utilise pour l'unification des scores.


In [ ]:
import random
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm
import os
from PIL import Image
import torch.nn.functional as F

seed = 42

random.seed(seed)                  # aleatoire integre de Python
np.random.seed(seed)               # NumPy
torch.manual_seed(seed)            # PyTorch (CPU)
torch.cuda.manual_seed(seed)       # PyTorch (un seul GPU)
torch.cuda.manual_seed_all(seed)   # PyTorch (tous les GPU)

# Assure un comportement deterministe
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
def sort_paths_by_number(path_list):
    """
    Sort based on the numerical values of the filenames in the path,
    assuming all filenames can be converted to integers.
    """
    def get_file_number(path):
        file_name = os.path.splitext(os.path.basename(path))[0]
        return int(file_name)

    path_list.sort(key=get_file_number)


In [ ]:
# importation du modele CLIP-ViT
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
%pip install --quiet git+https://github.com/openai/CLIP.git
import clip

model, preprocess = clip.load("ViT-B/16", device=DEVICE)
model.eval()  # evaluation zero-shot

In [ ]:
def infer(img_paths):
    """
    Compute L2‑normalized feature embeddings for a list of image file paths using the CLIP visual encoder.
    """
    # print(len(img_paths))  # pour le debogage
    embeddings = []
    for path in tqdm(img_paths):
        img = Image.open(path)
        x = preprocess(img)
        x = x.type(torch.float16).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            emb = model.visual.forward(x)

        embeddings.append(emb)

    embeddings = torch.cat(embeddings)
    embeddings = F.normalize(embeddings, p=2, dim=1)

    return embeddings

In [ ]:
def match_images(BASE_DATA_DIR, result_path):
    """
    For each query image in BASE_DATA_DIR/query, find its best matching image
    in BASE_DATA_DIR/gallery by computing cosine similarity of CLIP embeddings,
    then save 1‑based match indices to result_path as a .npy file.
    """
    QUERY_DIR = BASE_DATA_DIR / "query"
    NON_QUERY_DIR = BASE_DATA_DIR / "gallery"

    query_image_paths = list(QUERY_DIR.glob("*.png"))
    non_query_image_paths = list(NON_QUERY_DIR.glob("*.png"))

    query_image_paths_str = [str(p) for p in query_image_paths]
    non_query_image_paths_str = [str(p) for p in non_query_image_paths]

    sort_paths_by_number(query_image_paths_str)
    sort_paths_by_number(non_query_image_paths_str)

    # print(query_image_paths_str)  # pour le debogage

    query_embeddings = infer(query_image_paths_str)
    non_query_embeddings = infer(non_query_image_paths_str)
    distances = torch.mm(query_embeddings, non_query_embeddings.t())
    distances = (distances + 1.) / 2.

    topk_dists, topk_idxs = torch.topk(distances, 11, dim=1)  # distances a la forme (num_queries, num_non_queries)

    topk_dists, topk_idxs = topk_dists.cpu(), topk_idxs.cpu()

    matches_dists, matches_idxs = topk_dists[:, 1], topk_idxs[:, 1]
    matches_dists = matches_dists.cpu().numpy()
    matches_idxs = matches_idxs.cpu().numpy()

    for i in range(len(matches_idxs)):
        matches_idxs[i]+=1

    # print(matches_idxs)  # pour le debogage
    np.save(result_path, matches_idxs)  # sauvegarde au format .npy

In [ ]:
"""la soumission generee doit etre constituee de deux fichiers .npy"""
DATA_PATH = Path("Solution/")
OUTPUT_PATH = DATA_PATH / "Scoring"

match_images(DATA_PATH / "validation_set", OUTPUT_PATH / "submission_a.npy")
match_images(DATA_PATH / "test_set", OUTPUT_PATH / "submission_b.npy")

In [ ]:
import zipfile

files_to_zip = ['./Solution/Scoring/submission_a.npy', './Solution/Scoring/submission_b.npy']
zip_filename = 'submission.zip'

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} a ete cree avec succes !')